# SQL Refresh: Exploring the IMDb Database

**Course:** Database Applications Development  
**Medina County Career Center**

---

Welcome back to SQL! We've been away from databases for a bit, so today we're going to dust off our SQL skills with a brand new database — **IMDb movie data**.

This database has ~19,000 movies and TV series from 2010 to present, along with ratings, actors, directors, and crew. It's all titles you'll actually recognize.

**Today's goals:**
1. Reconnect to a database and remember how it works
2. Explore the structure of a new database
3. Practice SELECT, WHERE, ORDER BY, and GROUP BY
4. Find some movies and actors YOU care about

## Setup

First, let's import our libraries and connect to the database. Same pattern as before — nothing new here.

In [ ]:
import pandas as pd
import sqlite3

# Connect to the filtered IMDb database
dbPath = 'imdb_class_2010plus.db'
connection = sqlite3.connect(dbPath)
cursor = connection.cursor()

print(f'Connected to: {dbPath}')

---

## Part 1: What's in This Database?

Before we write any queries, let's see what tables we have. Remember `sqlite_master`? That's the system table that tells us what's inside the database.

In [ ]:
# List all tables in the database
tableQuery = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
tables = cursor.execute(tableQuery).fetchall()

print("Tables in imdb_class_2010plus.db:")
print("-" * 35)
for i, (tableName,) in enumerate(tables, 1):
    # Also get the row count for each table
    count = cursor.execute(f"SELECT COUNT(*) FROM {tableName}").fetchone()[0]
    print(f"  {i}. {tableName:25s} ({count:,} rows)")

---

## Part 2: SQL Refresher — The Basics

Let's warm up with queries you've seen before. Same SQL, new data.

### SELECT + WHERE

Remember: `SELECT` picks columns, `WHERE` filters rows.

Let's find some movies you know.

In [ ]:
# Find Inception
query = '''
    SELECT primaryTitle, startYear, genres, runtimeMinutes
    FROM title_basics
    WHERE primaryTitle = 'Inception'
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

#### New SQL Concept: `LIKE` with Wildcards

Sometimes you don't know the exact title — you just want to search for a keyword. That's what `LIKE` does.

- `LIKE '%Avengers%'` — finds any title that **contains** "Avengers" anywhere
- `LIKE 'The%'` — finds titles that **start with** "The"
- `LIKE '%Man'` — finds titles that **end with** "Man"

The `%` is a **wildcard** — it means "any number of characters." Think of it like the `*` in a Google search.

**Important:** `LIKE` is **case-sensitive** in SQLite by default for ASCII characters. So `'spider'` won't match `'Spider'`.

In [ ]:
# Find all Marvel movies (titles containing "Avengers")
query = '''
    SELECT primaryTitle, startYear, genres
    FROM title_basics
    WHERE primaryTitle LIKE '%Avengers%'
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

### Try It: Find Your Favorites

Change the query below to search for a movie or show YOU like. Try:
- An exact title: `WHERE primaryTitle = 'Your Movie'`
- A partial match: `WHERE primaryTitle LIKE '%keyword%'`

In [ ]:
# YOUR TURN: Search for a movie or show you like
query = '''
    SELECT primaryTitle, startYear, genres, runtimeMinutes
    FROM title_basics
    WHERE primaryTitle LIKE '%Spider%'
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

### ORDER BY + LIMIT

Remember: `ORDER BY` sorts results, `LIMIT` caps how many rows you get back.

#### New SQL Concepts: `JOIN`, `ON`, and Table Aliases

The next query pulls data from **two tables at once**. To do that, we need a few new tools:

**`JOIN ... ON`** — Combines rows from two tables where a column matches.
```sql
FROM title_basics b
JOIN title_ratings r ON b.tconst = r.tconst
```
This says: "Take each row from `title_basics`, find the matching row in `title_ratings` where the `tconst` values are the same, and glue them together."

**Table Aliases** — The `b` and `r` are shortcuts (aliases) so we don't have to type the full table name every time:
- `b` = `title_basics`
- `r` = `title_ratings`

Then we use `b.primaryTitle` instead of `title_basics.primaryTitle`. Same thing, less typing.

**Why do we need aliases?** Both tables have a column called `tconst`. If we just write `tconst`, SQL doesn't know which table we mean. The alias makes it clear: `b.tconst` vs `r.tconst`.

In [ ]:
# Top 10 highest-rated movies with at least 100,000 votes
# (We join title_basics and title_ratings to get both title info AND ratings)
query = '''
    SELECT b.primaryTitle, b.startYear, r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND r.numVotes > 100000
    ORDER BY r.averageRating DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print("Top 10 highest-rated movies (100K+ votes):")
print(df.to_string(index=False))

#### New SQL Concept: `IS NOT NULL`

Some movies are missing their runtime — the value is `NULL` (blank/unknown). If we're sorting by runtime, we want to skip those.

```sql
WHERE runtimeMinutes IS NOT NULL
```

This filters out any row where `runtimeMinutes` has no value. You can't use `= NULL` or `!= NULL` in SQL — it **must** be `IS NULL` or `IS NOT NULL`. That's just how SQL works.

In [ ]:
# Longest movies in the database
query = '''
    SELECT primaryTitle, startYear, runtimeMinutes, genres
    FROM title_basics
    WHERE runtimeMinutes IS NOT NULL
      AND titleType = 'movie'
    ORDER BY runtimeMinutes DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print("Longest movies:")
print(df.to_string(index=False))

### Try It: Your Own Ranking

Modify the query below to find YOUR top 10. Some ideas:
- Most popular movies from a specific year
- Highest rated TV series
- Shortest movies
- Movies from a specific genre (hint: `WHERE genres LIKE '%Horror%'`)

### Best genres to try first
- Action
- Comedy
- Drama
- Horror
- Thriller
- Sci-Fi
- Adventure
- Animation
- Crime

### Best genre combinations to try
- Action,Adventure
- Action,Comedy
- Action,Sci-Fi
- Action,Thriller
- Adventure,Fantasy
- Adventure,Sci-Fi
- Comedy,Drama
- Comedy,Romance
- Crime,Drama
- Horror,Thriller
- Mystery,Thriller

In [ ]:
# YOUR TURN: Create your own top 10 list
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND b.genres LIKE '%Adventure%'
      AND r.numVotes > 10000
    ORDER BY r.averageRating DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

---

### GROUP BY + Aggregations

Remember: `GROUP BY` groups rows together so you can use `COUNT()`, `AVG()`, `MAX()`, `MIN()`, `SUM()` on each group.

#### New SQL Concepts: `ROUND()` and `AVG()`

You already know `COUNT(*)` counts rows. Two more aggregation functions:

**`AVG(column)`** — Calculates the average (mean) of a numeric column.
```sql
AVG(r.averageRating)   -- average of all ratings in the group
```

**`ROUND(value, decimals)`** — Rounds a number to a specified number of decimal places.
```sql
ROUND(AVG(r.averageRating), 2)   -- average rounded to 2 decimal places
ROUND(AVG(r.numVotes), 0)        -- average rounded to 0 decimals (whole number)
```

Without `ROUND()`, you'd get numbers like `5.873291046...` which are hard to read. `ROUND(..., 2)` gives you `5.87`.

In [ ]:
# How many movies came out each year?
query = '''
    SELECT startYear, COUNT(*) as movieCount
    FROM title_basics
    WHERE titleType = 'movie'
      AND startYear >= 2015
    GROUP BY startYear
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print("Movies per year (2015+):")
print(df.to_string(index=False))

In [ ]:
# Average rating by year — are movies getting better or worse?
query = '''
    SELECT b.startYear, 
           COUNT(*) as movieCount,
           ROUND(AVG(r.averageRating), 2) as avgRating,
           ROUND(AVG(r.numVotes), 0) as avgVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND b.startYear >= 2015
    GROUP BY b.startYear
    ORDER BY b.startYear
'''

df = pd.read_sql_query(query, connection)
print("Movie ratings by year:")
print(df.to_string(index=False))

In [ ]:
# Which categories of people work on movies? (actor, director, writer, etc.)
query = '''
    SELECT category, COUNT(*) as totalCredits
    FROM title_principals
    GROUP BY category
    ORDER BY totalCredits DESC
'''

df = pd.read_sql_query(query, connection)
print("Credits by category:")
print(df.to_string(index=False))

### Try It: Your Own GROUP BY

Ideas to try:
- Count movies by genre (tricky — genres are comma-separated, so `LIKE '%Action%'` with GROUP BY year)
- Find the year with the most TV series
- Average runtime by titleType (movie vs tvSeries)

In [ ]:
# YOUR TURN: Write a GROUP BY query
query = '''
    SELECT titleType, 
           COUNT(*) as count,
           ROUND(AVG(runtimeMinutes), 1) as avgRuntime
    FROM title_basics
    WHERE runtimeMinutes IS NOT NULL
    GROUP BY titleType
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

---

## Part 3: Connecting Tables (JOINs)

The real power of this database comes when we connect tables together. We've been doing this already with `JOIN` — let's make it explicit.

#### New SQL Concepts: `IN ()` and `DISTINCT`

Two more tools you'll see in the JOIN queries below:

**`IN (value1, value2, ...)`** — A shortcut for multiple `OR` conditions.
```sql
WHERE p.category IN ('actor', 'actress')
```
This is the same as writing:
```sql
WHERE p.category = 'actor' OR p.category = 'actress'
```
Much cleaner when you have several values to check!

**`DISTINCT`** — Removes duplicate values from a result.
```sql
COUNT(DISTINCT p.tconst)   -- counts unique movie IDs only
```
Without `DISTINCT`, if an actor has two credits on the same movie (like appearing in two roles), it would count that movie twice. `DISTINCT` makes sure each movie is counted only once.

In [ ]:
# Who are the most prolific actors in this database?
query = '''
    SELECT n.primaryName, COUNT(DISTINCT p.tconst) as movieCount
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    WHERE p.category IN ('actor', 'actress')
    GROUP BY n.nconst
    ORDER BY movieCount DESC
    LIMIT 15
'''

df = pd.read_sql_query(query, connection)
print("Most prolific actors (by number of titles):")
print(df.to_string(index=False))

In [ ]:
# What movies has a specific actor been in?
# Let's try Nicolas Cage
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, r.averageRating
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE n.primaryName = 'Nicolas Cage'
      AND p.category = 'actor'
      AND b.titleType = 'movie'
    ORDER BY b.startYear DESC
'''

df = pd.read_sql_query(query, connection)
print(f"Nicolas Cage movies ({len(df)} total):")
print(df.to_string(index=False))

### Try It: Look Up YOUR Favorite Actor

Change the name in the query below to find movies for an actor you like.

In [ ]:
# YOUR TURN: Pick an actor and find their movies
query = '''
    SELECT b.primaryTitle, b.startYear, r.averageRating, p.characters
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE n.primaryName = 'Zendaya'
      AND p.category IN ('actor', 'actress')
    ORDER BY b.startYear DESC
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

In [ ]:
# Who starred in a specific movie?
query = '''
    SELECT n.primaryName, p.category, p.characters
    FROM title_basics b
    JOIN title_principals p ON b.tconst = p.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE b.primaryTitle = 'Inception'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''

df = pd.read_sql_query(query, connection)
print("Cast & director of Inception:")
print(df.to_string(index=False))

### Try It: Who's in YOUR Favorite Movie?

Change the title below to look up the cast of a movie you love.

In [ ]:
# YOUR TURN: Look up the cast of a movie
query = '''
    SELECT n.primaryName, p.category, p.characters
    FROM title_basics b
    JOIN title_principals p ON b.tconst = p.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE b.primaryTitle = 'The Batman'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

---

## Wrap-Up

**Today we refreshed:**
- `SELECT` / `WHERE` / `ORDER BY` / `LIMIT` — picking and filtering data
- `GROUP BY` with `COUNT()`, `AVG()` — summarizing data
- `JOIN` — connecting tables together

**The IMDb database has 5 tables:**
- `title_basics` and `title_ratings` hold movie/show info
- `name_basics` holds people info
- `title_principals` and `title_crew_person` connect people to titles

**Next up:** Your task notebook — explore this database on your own!

In [ ]:
# Clean up
connection.close()
print("Database connection closed.")